# Analysis Template: Michaelis–Menten

Updated 5/22/26, Nicholas Freitas

In [ ]:
#enables autoreloding of modules
%load_ext autoreload
%autoreload 2

from mercury.db_api.mercury_db_api import LocalMercuryDBAPI
from mercury.analysis.experiment import MercuryExperiment
from mercury.analysis import plot
from mercury.db_api.units import units
from pathlib import Path

#enable inline plotting of matplotlib figures
%matplotlib inline

#set the figure format to SVG
%config InlineBackend.figure_format = 'svg'

## 1. Connect DB Api

First, we load your data. Make sure to use the correct path, filenames, and column names for standard and kinetics data.

In [ ]:
### PARAMETERS:
EGFP_SLOPE = 91900.03 / 9 * (units.RFU / units.nM)

# Print our current working directory, for convenience:
print('Currently in: ', Path.cwd())

# Make sure to use the correct relative path!
root = './your_data_folder/' 

db_conn = LocalMercuryDBAPI(
    standard_curve_data_path= root + 'standard_data.csv.bz2',
    standard_name="NADPH", 
    standard_substrate="NADPH", 
    standard_units=units.uM,
    standard_concentration_col="concentration",  # must match the column name in your standard curve CSV
    
    kinetic_data_path= root + 'kinetics_data_merged.csv.bz2',
    kinetic_name="ADP", 
    kinetic_substrate="ADP", 
    kinetic_units=units.uM,
    kinetic_concentration_col="LMET_conc",       # must match the column name in your kinetics CSV
    
    time_units=units.s,
    button_quant_data_path= root + 'button_quant.csv'
)

mercury_experiment = MercuryExperiment(db_conn)

## 2. Enzyme Quant

In [ ]:
from mercury.analysis.transform import transform_data

button_concentrations = transform_data(
    data_objs = [mercury_experiment.get_run('button_quant')],              # e.g. a Data2D or Data3D or Data4D instance
    expr=f'(a.luminance / slope)',             # e.g. "(luminance - intercept) / slope"
    expression_vars={'slope': EGFP_SLOPE},
    output_name='concentration'       # name of the new field, e.g. "concentration"
)

mercury_experiment.set_run('enzyme_concentrations', button_concentrations)     # save the fit results

In [ ]:
plot.plot_enzyme_concentration_chip(mercury_experiment, analysis_name='enzyme_concentrations')

## 2. Product Standards

In [ ]:
from mercury.analysis.fit import fit_luminance_vs_concentration

standard_experiment_data = mercury_experiment.get_run('NADPH')       # retrieve our raw data
standard_fits = fit_luminance_vs_concentration(standard_experiment_data)    # perform a fit
mercury_experiment.set_run('NADPH_standard', standard_fits)              # save the fit results

What if we need to mask out bad datapoints?
We can select them and create a mask. Then, we apply the mask as seen below:

In [ ]:
# # Here, I'm showing how to create a custom mask to ignore bad std curve timepoints

# from mercury.analysis.fit import fit_luminance_vs_concentration
# from mercury.analysis.filter import make_custom_mask
# import numpy as np

# standard_experiment_data = mercury_experiment.get_run('NADPH')       # retrieve our raw data
# standard_experiment_data.dep_var.shape # (8,1, 1792, 1)

# # # let's mask out the second-to-last concentration
# mask = np.ones(standard_experiment_data.dep_var.shape) * True
# mask[-2, :,:,:] = False
# mask = mask.astype(bool)
# custom_mask = make_custom_mask(standard_experiment_data, mask, info='final_timepoint_mask')
# mercury_experiment.set_run('final_timepoint_mask', custom_mask)

# mercury_experiment.apply_mask(run_name='NADPH', 
#                             dep_variables = ['luminance'], 
#                             save_as = 'NADPH_masked',
#                             mask_names = ['final_timepoint_mask'])

# standard_experiment_data = mercury_experiment.get_run('NADPH_masked')       # retrieve our raw data
# standard_fits = fit_luminance_vs_concentration(standard_experiment_data)    # perform a fit
# mercury_experiment.set_run('NADPH_standard', standard_fits)              # save the fit results


In [ ]:
plot.plot_standard_curve_chip(mercury_experiment, 'NADPH_standard', 'NADPH_masked')

## 3. Fit Initial Rates

First, we convert from RFU to product concentration, using our standard curves:

In [ ]:
# Calculate product concentrations from RFU data:
product_concentrations = transform_data(
    data_objs = [mercury_experiment.get_run('ADP'), mercury_experiment.get_run('NADPH_standard')],
    expr=f'(a.luminance - b.intercept) / b.slope',             # e.g. "(luminance - intercept) / slope"
    output_name='concentration'       # name of the new field, e.g. "concentration"
)
 
mercury_experiment.set_run('kinetics_ADP_conc', product_concentrations)

Pay close attention to your start and end timepoint for your fit window.

By convention, we make sure not to use points where the product concentration is greater than 10% of the maximum, so we stay in the initial reaction phase.

In [ ]:
from mercury.analysis.fit import fit_concentration_vs_time

kinetics_concentrations = mercury_experiment.get_run('kinetics_ADP_conc')       # retrieve our raw data

# Here we perform the fit. Decide the optimal start and end timepoints,
# And the max reaction percent.
kinetics_fits, fit_points_mask = fit_concentration_vs_time(kinetics_concentrations, 
                                            start_timepoint = 1, 
                                            end_timepoint=10,
                                            max_reaction_percent=10) # default 10%      

mercury_experiment.set_run('kinetics_ADP_conc_fits', kinetics_fits)                # save the fit results
mercury_experiment.set_run('fit_points_mask', fit_points_mask)


Need to use custom fit windows for each concentration?

In [ ]:
## Example for overriding the fit windows per concentration:
# kinetics_fits = fit_concentration_vs_time(kinetics_concentrations,
#                                            fit_windows_per_concentration=
#                                            {0.5: (1, 20),
#                                             1.0: (1, 6),
#                                             2.0: (1, 6),
#                                             4.0: (1, 6),
#                                             8.0: (1, 6),
#                                             16.0: (1, 6),
#                                             32.0: (1, 6),
#                                             64.0: (1, 6),
#                                             128.0: (1, 6),
#                                             256.0: (1, 20)})

In [ ]:
plot.plot_initial_rates_chip(mercury_experiment, analysis_name='kinetics_ADP_conc_fits', experiment_name='kinetics_ADP_conc',
                            fit_points_mask="fit_points_mask",
                            
                            # To zoom in on the graphs, you can set x and y limits:
                            plot_xmin=None, plot_xmax=None,
                            plot_ymin=None, plot_ymax=None)#, remove_0_point=True) 

## 3.5 Background Rate Subtraction

In this section, we find a background rate by looking at all of our buffer wells, and taking the 25th percentile of those slopes.

Then, we subtract that from all of our chambers.

In [ ]:
from mercury.analysis.filter import filter_by_sample_id

# Here, we create a mask of the data to only have the buffer wells. 
buffer_mask =  filter_by_sample_id(kinetics_fits, 
                                    sample_ids=['buffer'])
mercury_experiment.set_run('buffer_mask', buffer_mask)

# Apply the masks to the fits and save the results
mercury_experiment.apply_mask(run_name='kinetics_ADP_conc_fits', 
                            dep_variables = ['slope', 'intercept', 'r_squared'], 
                            save_as = 'buffer_wells',
                            mask_names = ['buffer_mask'])

# Now, we'll do processing on the buffer wells:
buffer_wells = mercury_experiment.get_run('buffer_wells')

# In each well, we'll store the 25th percentile of all buffer slopes.
buffer_wells_lower_quartile = transform_data(
    data_objs = [buffer_wells],              # e.g. a Data2D or Data3D or Data4D instance
    expr=f'np.nanpercentile(a.device.slope, 25, axis=1)',             # e.g. "(luminance - intercept) / slope"
    output_name='slope_lower_quartile'       # name of the new field, e.g. "slope"
)

mercury_experiment.set_run('buffer_wells_lower_quartile', buffer_wells_lower_quartile)     # save the fit results

# Now, we'll transform the data to subtract the background rates, per-concentration:
corrected_initial_rates = transform_data(
    data_objs = [mercury_experiment.get_run('kinetics_ADP_conc_fits'), mercury_experiment.get_run('buffer_wells_lower_quartile')],              # e.g. a Data2D or Data3D or Data4D instance
    expr=f'(a.chamber.slope - b.chamber.slope_lower_quartile)',             # e.g. "(luminance - intercept) / slope"
    output_name='slope',       # name of the new field, e.g. "slope"
    keep_existing = True   # keep the existing fields (slope, intercept, r_squared)
)

mercury_experiment.set_run('kinetics_ADP_conc_fits_bgsub', corrected_initial_rates)     # save the fit results


## 4. Filter initial rates

First, we make masks to filter our data:

In [ ]:
# These are the "Masks" we use to select which data we want.

# In this example we use:
# standard_curve_r2_cutoff = 0.9,
# expression_threshold = 1.0nM,
# initial_rate_R2_threshold = 0.9, 
# positive_initial_slope_filter = True

from mercury.analysis.filter import filter_expression_cutoff, filter_initial_rates_positive_cutoff, filter_initial_rates_r2_cutoff, filter_standard_curve_r2_cutoff

kinetics_fits = mercury_experiment.get_run('kinetics_ADP_conc_fits_bgsub')       # retrieve our raw data

# Initial Rates:
initial_rates_r2_mask =         filter_initial_rates_r2_cutoff(kinetics_fits, r2_cutoff=0.9) # 0.9 R2
initial_rates_positive_mask =   filter_initial_rates_positive_cutoff(kinetics_fits)          # positive slope

# Standard Curve:
standard_curve_r2_mask =        filter_standard_curve_r2_cutoff(standard_fits, kinetics_fits, r2_cutoff=0.9) # 0.9 R2

# Expression:
expression_mask =               filter_expression_cutoff(button_concentrations, kinetics_fits, expression_cutoff=1) # 1 nM

# Save the masks to the experiment
mercury_experiment.set_run('initial_rates_r2_mask', initial_rates_r2_mask)
mercury_experiment.set_run('initial_rates_positive_mask', initial_rates_positive_mask)
mercury_experiment.set_run('standard_curve_r2_mask', standard_curve_r2_mask)
mercury_experiment.set_run('expression_mask', expression_mask)



Before we apply the masks, let's look at them to see what's making it through. If we're not happy, we can go up a cell and adjust the parameters.

In [ ]:
# Plot the masks (Do one at a time)
plot.plot_mask_chip(mercury_experiment, mask_name='initial_rates_r2_mask')
#plot.plot_mask_chip(mercury_experiment, mask_name='initial_rates_positive_mask')
#plot.plot_mask_chip(mercury_experiment, mask_name='standard_curve_r2_mask')
#plot.plot_mask_chip(mercury_experiment, mask_name='expression_mask')

Finally, let's apply the masks:

In [ ]:
# Apply the masks to the fits and save the results
mercury_experiment.apply_mask(run_name='kinetics_ADP_conc_fits_bgsub', 
                            dep_variables = ['slope', 'intercept'], 
                            save_as = 'kinetics_ADP_conc_fits_masked',
                            mask_names = ['initial_rates_r2_mask', 'initial_rates_positive_mask', 'standard_curve_r2_mask', 'expression_mask'])


Need to manually mask out more points? Use our GUI to select them.

In [ ]:
# from mercury.analysis.interactive import ImageMaskPicker

# # You can pass multiple images to switch between them!
# # Only works with "summary_image" outputs from processing.
# images = {
#     'Standard Curve Image': './path/to/your/image.tif',
# }

# picker = ImageMaskPicker(images, n_cols=32, n_rows=56)
# picker.show()

When you're done, make sure to apply the new mask:

In [ ]:
# manual_qc_mask = picker.get_mask(kinetics_fits, mask_name='manual_qc_mask')
# mercury_experiment.set_run('manual_qc_mask', manual_qc_mask)
# #plot.plot_mask_chip(mercury_experiment, mask_name='manual_qc_mask')

# mercury_experiment.apply_mask(run_name='kinetics_ADP_conc_fits_masked', 
#                             dep_variables = ['slope', 'intercept', 'r_squared'],
#                             mask_names=['manual_qc_mask'],
#                             save_as = 'kinetics_ADP_conc_fits_masked') # Overwrite the previous masked data

Let's delete chambers that don't have enough concentrations that passed filtering:

In [ ]:
# Now, we'll delete any wells that have fewer than 5 initial rates:
from mercury.analysis.filter import filter_number_concentrations

kinetics_fits = mercury_experiment.get_run('kinetics_ADP_conc_fits_masked')       # retrieve our raw data
number_initial_rates_mask = filter_number_concentrations(kinetics_fits, min_concentrations=5, var_to_check='slope')
mercury_experiment.set_run('number_initial_rates_mask', number_initial_rates_mask)
mercury_experiment.apply_mask(run_name='kinetics_ADP_conc_fits_masked', 
                            dep_variables = ['slope', 'intercept', 'r_squared'],
                            mask_names=['number_initial_rates_mask'],
                            save_as = 'kinetics_ADP_conc_fits_masked') # Overwrite the previous masked data

Let's plot our rates here.

Note! Because we subtracted the background rate, our slope lines will fall below the datapoints! That is expected.

In [ ]:
plot.plot_initial_rates_chip(experiment=mercury_experiment, analysis_name='kinetics_ADP_conc_fits_masked', experiment_name='kinetics_ADP_conc', fit_points_mask="fit_points_mask")#, remove_0_point=True) # plot the initial rates

Now, the slopes vs concentration:

In [ ]:
plot.plot_initial_rates_vs_concentration_chip(experiment=mercury_experiment, analysis_name='kinetics_ADP_conc_fits_masked') # plot the initial rates

## 5. Fit MM Constant:

In [ ]:
from mercury.analysis.fit import fit_initial_rates_vs_concentration_with_function, mm_model, inhibition_model

kinetics_fits_masked = mercury_experiment.get_run('kinetics_ADP_conc_fits_masked')       # retrieve our raw data
MM_fits, MM_pred_data = fit_initial_rates_vs_concentration_with_function(data = kinetics_fits_masked,
                                model_func = mm_model)
mercury_experiment.set_run('kinetics_ADP_MM_fits', MM_fits)                # save the fit results
mercury_experiment.set_run('kinetics_ADP_MM_pred_data', MM_pred_data)    # save the predicted data

We've fit our Michaelis-Menten model. Now, let's filter out poor fits.

First, we filter by R2, then we make sure we have enough replicates.

In [ ]:
from mercury.analysis.filter import filter_r2_cutoff, filter_number_replicates
 
MM_fits = mercury_experiment.get_run('kinetics_ADP_MM_fits')

# Change your R2 cutoff!
MM_fits_mask = filter_r2_cutoff(MM_fits, r2_cutoff=0.8) 
MM_fits_replicate_mask = filter_number_replicates(MM_fits, min_replicates=5, var_to_check='K_m')

mercury_experiment.set_run('MM_R2_mask', MM_fits_mask)  # save the fit results
mercury_experiment.set_run('MM_replicate_mask', MM_fits_replicate_mask)

plot.plot_mask_chip(experiment=mercury_experiment, mask_name='MM_replicate_mask')

Once we have a good filter, we apply the mask to the data and plot:

In [ ]:
# Apply the mask to the fits and save the results:
mercury_experiment.apply_mask(run_name='kinetics_ADP_MM_fits', 
                            dep_variables = ['v_max', 'K_m', 'r_squared'], 
                            save_as = 'kinetics_ADP_MM_fits_masked',
                            mask_names = ['MM_R2_mask', 'MM_replicate_mask'])

In [ ]:
plot.plot_MM_chip(experiment=mercury_experiment, analysis_name='kinetics_ADP_conc_fits_masked', 
                                model_fit_name='kinetics_ADP_MM_fits_masked',
                                model_pred_data_name='kinetics_ADP_MM_pred_data',
                                )#x_log=True) # plot the initial rates

# 6. Divide by $[E]$ to get $k_{cat}$, $V_0/[E]$

In [ ]:
# This is the same data we made above, we're just going to 
# divide everything by enzyme concentration.

# enzyme concentration:
enzyme_concentrations = mercury_experiment.get_run("enzyme_concentrations")

# masked slopes vs conc:
masked_slopes_vs_conc = mercury_experiment.get_run("kinetics_ADP_conc_fits_masked")
# MM masked fits:
masked_fits = mercury_experiment.get_run("kinetics_ADP_MM_fits_masked")
# pred data from MM fits:
pred_data = mercury_experiment.get_run("kinetics_ADP_MM_pred_data")

# Get V_0/[E]:
masked_V0_div_E_vs_conc = transform_data(
    data_objs = [masked_slopes_vs_conc, enzyme_concentrations],
    expr=f'a_slope / b_concentration',             # e.g. "(luminance - intercept) / slope"
    output_name='V0_div_E'       # name of the new field, e.g. "concentration"
)

# Get V_max/[E] (kcat)
masked_fits_with_kcat = transform_data(
    data_objs = [masked_fits, enzyme_concentrations],
    expr=f'a_v_max / b_concentration',             # e.g. "(luminance - intercept) / slope"
    output_name='kcat',       # name of the new field, e.g. "concentration",
    keep_existing = True
)

# # Do the same for pred data:
pred_data_div_E = transform_data(
    data_objs = [pred_data, enzyme_concentrations],
    expr=f'a_y_pred / b_concentration',             # e.g. "(luminance - intercept) / slope"
    output_name='pred_V0_div_E',       # name of the new field, e.g. "concentration",
)

# Add to experiment
mercury_experiment.set_run('masked_V0_div_E_vs_conc', masked_V0_div_E_vs_conc)
mercury_experiment.set_run('masked_fits_with_kcat', masked_fits_with_kcat)
mercury_experiment.set_run('pred_data_div_E', pred_data_div_E)

## Remove Outliers

If we want, we can manually remove some crazy outliers like so:

In [ ]:
#### Here, I show how to manually remove some crazy outliers (usually not necessary)
# MM_outlier_mask_manual = transform_data(
#     data_objs = [mercury_experiment.get_run('masked_fits_with_kcat')],
#     expr=f'a.chamber.K_m.magnitude < 1000',
#     output_name='mask'
# )

# mercury_experiment.set_run('MM_outlier_mask_manual', MM_outlier_mask_manual)
# #plot.plot_mask_chip(mercury_experiment, mask_name='MM_outlier_mask_manual')

#### Apply the mask to the fits and save the results:
# mercury_experiment.apply_mask(run_name='masked_fits_with_kcat',
#                             dep_variables = ['v_max', 'K_m', 'r_squared', 'kcat'], 
#                             save_as = 'masked_fits_with_kcat_filtered',
#                             mask_names = ['MM_outlier_mask_manual'])

Generally, we filter by z-score. That is, replicates with parameters (k_cat, K_M) that are Z standard deviations above the sample mean will be discarded.

In [ ]:
# NaN all chamber values where K_m is 1.5 standard deviations away from the mean
z_score_threshold = 1.5

# First, we make a boolean mask of all chambers that are within 3 standard deviations of the mean K_m across the entire device.
MM_outlier_mask_KM = transform_data(
    data_objs = [mercury_experiment.get_run('masked_fits_with_kcat')], # or use masked_fits_with_kcat_filtered
    expr=f'~(np.abs(a.chamber.K_m - np.nanmean(a.sample.K_m)) > z_score_threshold * np.nanstd(a.sample.K_m))',
    expression_vars={'z_score_threshold': z_score_threshold},
    output_name='mask'
)

MM_outlier_mask_kcat = transform_data(
    data_objs = [mercury_experiment.get_run('masked_fits_with_kcat')], # or use masked_fits_with_kcat_filtered
    expr=f'~(np.abs(a.chamber.kcat - np.nanmean(a.sample.kcat)) > z_score_threshold * np.nanstd(a.sample.kcat))',
    expression_vars={'z_score_threshold': z_score_threshold},
    output_name='mask'
)

MM_outlier_mask_expression = transform_data(
    data_objs = [mercury_experiment.get_run('enzyme_concentrations')], # or use masked_fits_with_kcat_filtered
    expr=f'~(np.abs(a.chamber.concentration - np.nanmean(a.sample.concentration)) > z_score_threshold * np.nanstd(a.sample.concentration))',
    expression_vars={'z_score_threshold': z_score_threshold},
    output_name='mask'
)

# And them all together:
MM_outlier_mask = transform_data(
    data_objs = [MM_outlier_mask_KM, MM_outlier_mask_kcat, MM_outlier_mask_expression],
    expr=f'a.mask.magnitude & b.mask.magnitude & c.mask.magnitude',
    output_name='mask'
)

mercury_experiment.set_run('MM_outlier_mask', MM_outlier_mask)
plot.plot_mask_chip(mercury_experiment, mask_name='MM_outlier_mask')

# Apply the mask to the fits and save the results:
mercury_experiment.apply_mask(run_name='masked_fits_with_kcat',
                            dep_variables = ['v_max', 'K_m', 'r_squared', 'kcat'], 
                            save_as = 'masked_fits_with_kcat_filtered',
                            mask_names = ['MM_outlier_mask'])

# Also apply the mask to the slope data:
mercury_experiment.apply_mask(run_name='masked_V0_div_E_vs_conc',
                            dep_variables = ['V0_div_E'], 
                            save_as = 'masked_V0_div_E_vs_conc_filtered',
                            mask_names = ['MM_outlier_mask'])
                            

Plot the results:

In [ ]:
# Expects 'V0_div_E'
plot.plot_MM_div_E_chip(experiment=mercury_experiment, analysis_name='masked_V0_div_E_vs_conc_filtered', 
                                model_fit_name='masked_fits_with_kcat_filtered',
                                dep_var_name='V0_div_E',
                                )

## 7. Export to CSV

All of our data is sent to CSV for further processing.

In [ ]:
mercury_experiment.export_MM_chamber_data(run_name='masked_fits_with_kcat_filtered', enzyme_concentration_run_name='enzyme_concentrations', file_name='MM_chamber_data.csv')
mercury_experiment.export_MM_sample_data(run_name='masked_fits_with_kcat_filtered', enzyme_concentration_run_name='enzyme_concentrations', file_name='MM_sample_data.csv')

Next, we output a PDF with the fit Michaelis-Menten parameters for each sample:

In [ ]:
mercury_experiment.export_mm_subplots_by_sample(
                           analysis_name='masked_V0_div_E_vs_conc_filtered',
                           model_fit_name='masked_fits_with_kcat_filtered',
                           dep_var_label='V0_div_E',
                           export_path = 'mm_sample_subplots.pdf')

Finally, we can publish an end-to-end plot for each sample, to show the whole experiment.

You can run this per-sample. If you want to export all samples, this will take a long time! (~20 mins)

Make sure to change the labels in mask_runs to match your filters! This makes your graphs interpretable.

In [ ]:
mercury_experiment.export_end_to_end_summary_by_sample(
    export_dir='./all_samples_visual_summary',

    # Optional: remove this next line to process all samples!
    sample_ids=['Q339D'], 

    # We need to link to our original experimental images. The function will grab summary_images.
    button_quant_csv=root+'button_quant.csv',
    standard_curve_csv=root+'standard_data.csv.bz2',
    kinetics_csv=root+'kinetics_data.csv.bz2',

    # Label our masks. Change these descriptions to the values you used above!
    mask_runs={
        'expression_mask': 'Excluded: Expression < 10nM',
        'standard_curve_r2_mask': 'Excluded: Std Curve R2 < 0.5',
        'number_initial_rates_mask': 'Excluded: < 5 Kept Concentrations',
        'MM_outlier_mask': 'Excluded: MM Parameter Outlier',
        'MM_replicate_mask': 'Excluded: < 5 Kept Replicates',
        'initial_rates_r2_mask': 'Poor Init Rate R2',
        'initial_rates_positive_mask': 'Negative Init Rate',
    },

    dpi=150
)

Optionally, we can drill into the data by showing a plot for every individual chamber. This takes a while to output:

In [ ]:
# # This takes long!
# # Saving the figure takes around 2-5 minutes...
# mercury_experiment.export_mm_subplots_by_chamber(
#                            analysis_name='masked_V0_div_E_vs_conc_filtered',
#                            model_fit_name='masked_fits_with_kcat_filtered',
#                            dep_var_label='V0_div_E',
#                            export_path = 'mm_chamber_subplots.pdf')